In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
os.listdir('/content/drive/MyDrive/E1_TicketsTI')

['synthetic_it_support_tickets.csv']

In [4]:
import os
ruta_csv = '/content/drive/MyDrive/E1_TicketsTI/synthetic_it_support_tickets.csv'
os.path.exists(ruta_csv)


True

## 01. Fuente y carga del dataset

**Fuente:** Kaggle — "IT Support Tickets" (synthetic SaaS helpdesk), autor ahsanneural.

**Enlace:** https://www.kaggle.com/datasets/ahsanneural/synthetic-it-support-tickets

**Descripción:** Dataset sintético de 100k tickets de soporte técnico de una empresa SaaS,
con información de prioridad, SLA y satisfacción del cliente (CSAT).

In [6]:
import pandas as pd

df = pd.read_csv(ruta_csv)
df.head()

,ticket_id,created_at,customer_id,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,has_attachment,platform,region
0,TCKT_000001,2024-01-31T05:14:27,CUST_00861,individual,email,data_export,account_access,low,resolved,standard,I cannot log in; the system says my password i...,Sorry to hear you're having trouble accessing ...,Reset account credentials and confirmed succes...,36.53,0,very_negative,1,0,android,EU
1,TCKT_000002,2024-10-20T06:15:49,CUST_00770,individual,in_app,billing,security_concern,medium,closed_no_action,standard,I noticed a suspicious login on my account.,We take security very seriously. Our team is r...,Ticket closed without further action after no ...,238.32,0,neutral,3,0,web,NaN
2,TCKT_000003,2024-06-18T21:35:54,CUST_02559,small_business,chat,api_integration,bug,low,in_progress,standard,The api integration feature is not saving my c...,Thanks for reporting this bug. We will look in...,NaN,NaN,0,neutral,3,0,android,MEA
3,TCKT_000004,2025-12-25T15:59:52,CUST_03557,education,chat,analytics_dashboard,account_access,medium,in_progress,standard,I cannot log in; the system says my password i...,Sorry to hear you're having trouble accessing ...,NaN,NaN,0,positive,5,1,android,LATAM
4,TCKT_000005,2023-08-27T16:08:33,CUST_09556,enterprise,phone_transcript,login_auth,billing_problem,low,resolved,gold,My invoice amount is incorrect compared to the...,Thanks for reaching out about the billing issu...,Adjusted the invoice and issued a refund where...,61.32,0,very_negative,2,0,web,NaN


## 02. Reconocimiento del dataset

**Dimensiones:** 100,000 filas × 20 columnas.

**Descripción de las variables principales:**

- **Identificadores:** `ticket_id` (identifica cada ticket), `customer_id` (identifica al cliente que reportó el ticket).

- **Datos del ticket:** `created_at` (fecha de creación), `channel` (canal de entrada: email, chat, in_app, phone_transcript, web_form), `product_area` (área del producto afectada), `issue_type` (tipo de problema), `priority` (low/medium/high/urgent), `status` (open/in_progress/on_hold/resolved/closed_no_action).

- **Datos del cliente:** `customer_segment` (individual, small_business, education, enterprise, non_profit), `sla_plan` (standard/gold/platinum), `region` (EU/MEA/LATAM/APAC).

- **Contenido textual:** `initial_message`, `agent_first_reply`, `resolution_summary`.

- **Métricas de resolución:** `resolution_time_hours` (horas hasta resolver), `reopened` (si fue reabierto), `csat_score` (satisfacción del cliente, 0-5), `customer_sentiment`.

- **Otros:** `has_attachment`, `platform` (android/ios/web/desktop_app/api_client).

**Fuente de esta descripción:** inspección directa de las columnas del CSV con `df.columns` y `df.dtypes`.

In [16]:
# "Esto te confirma cuántas filas y columnas tiene"
df.shape


(100000, 20)

In [17]:
# "Esto te muestra el tipo de dato de cada columna (texto, número entero, decimal) y cuántos valores no nulos tiene cada una"
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ticket_id              100000 non-null  object 
 1   created_at             100000 non-null  object 
 2   customer_id            100000 non-null  object 
 3   customer_segment       100000 non-null  object 
 4   channel                100000 non-null  object 
 5   product_area           100000 non-null  object 
 6   issue_type             100000 non-null  object 
 7   priority               100000 non-null  object 
 8   status                 100000 non-null  object 
 9   sla_plan               100000 non-null  object 
 10  initial_message        100000 non-null  object 
 11  agent_first_reply      100000 non-null  object 
 12  resolution_summary     60113 non-null   object 
 13  resolution_time_hours  60113 non-null   float64
 14  reopened               100000 non-nul

In [18]:
# ¿Hay tickets duplicados?
df['ticket_id'].duplicated().sum()

np.int64(0)

In [19]:
# ¿Cuántos valores nulos hay por columna?
df.isna().sum()

,0
ticket_id,0
created_at,0
customer_id,0
customer_segment,0
channel,0
product_area,0
issue_type,0
priority,0
status,0
sla_plan,0


In [20]:
# ¿Cuántos clientes distintos hay vs. cuántos tickets?
df['customer_id'].nunique(), df['ticket_id'].nunique()

(9999, 100000)

In [21]:
# ¿Qué valores existen en las columnas categóricas clave?
for col in ['priority', 'status', 'sla_plan', 'customer_segment']:
    print(col, '->', df[col].unique())

priority -> ['low' 'medium' 'urgent' 'high']
status -> ['resolved' 'closed_no_action' 'in_progress' 'on_hold' 'open']
sla_plan -> ['standard' 'gold' 'platinum']
customer_segment -> ['individual' 'small_business' 'education' 'enterprise' 'non_profit']


In [22]:
# ¿Un mismo cliente aparece siempre con el mismo segmento/sla/region, o varía?
inconsistencias = df.groupby('customer_id')[['customer_segment', 'sla_plan']].nunique()
(inconsistencias > 1).sum()

,0
customer_segment,9987
sla_plan,9806


## 05. Reglas derivadas de los hallazgos

| Hallazgo | Regla para el sistema |
|---|---|
| `ticket_id` sin duplicados | Debe ser PRIMARY KEY de la tabla tickets. |
| `resolution_time_hours` y `resolution_summary` solo existen si el ticket está resuelto/cerrado | No son obligatorios al crear el ticket, pero el sistema debe impedir guardar un tiempo de resolución en un ticket que sigue open/in_progress/on_hold. |
| `region` con ~20% de nulos, sin relación con el canal | Es un dato opcional real; el sistema debe permitir NULL pero no debe excluir tickets de reportes solo por esto — se debe manejar explícitamente ("región no especificada"). |
| `customer_segment`, `sla_plan`, `region` varían por ticket para un mismo cliente | Estos NO son atributos fijos del cliente: se guardan a nivel de ticket (foto del momento), no en la tabla clientes. |
| `priority`, `status`, `sla_plan`, `customer_segment` sin errores de tipeo | Deben restringirse con CHECK a la lista exacta de valores válidos, para que un futuro registro no introduzca un valor inválido (ej. "urgentt"). |
| Cada ticket pertenece a un cliente existente | `customer_id` en tickets debe ser FOREIGN KEY hacia clientes — no se debe poder crear un ticket de un cliente que no existe. |

In [23]:
import sqlite3

ruta_db = '/content/drive/MyDrive/E1_TicketsTI/e1_tickets.db'
conn = sqlite3.connect(ruta_db)
conn.execute('PRAGMA foreign_keys = ON;')

In [24]:
conn.executescript('''
CREATE TABLE IF NOT EXISTS clientes (
    customer_id TEXT PRIMARY KEY
);

CREATE TABLE IF NOT EXISTS tickets (
    ticket_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL,
    created_at TEXT NOT NULL,
    customer_segment TEXT NOT NULL CHECK (customer_segment IN
        ('individual','small_business','education','enterprise','non_profit')),
    sla_plan TEXT NOT NULL CHECK (sla_plan IN ('standard','gold','platinum')),
    region TEXT CHECK (region IS NULL OR region IN ('EU','MEA','LATAM','APAC')),
    channel TEXT NOT NULL CHECK (channel IN
        ('email','chat','in_app','phone_transcript','web_form')),
    product_area TEXT NOT NULL,
    issue_type TEXT NOT NULL,
    priority TEXT NOT NULL CHECK (priority IN ('low','medium','high','urgent')),
    status TEXT NOT NULL CHECK (status IN
        ('open','in_progress','on_hold','resolved','closed_no_action')),
    initial_message TEXT,
    agent_first_reply TEXT,
    platform TEXT,
    has_attachment INTEGER CHECK (has_attachment IN (0,1)),
    reopened INTEGER CHECK (reopened IN (0,1)),
    FOREIGN KEY (customer_id) REFERENCES clientes(customer_id)
);

CREATE TABLE IF NOT EXISTS resoluciones (
    ticket_id TEXT PRIMARY KEY,
    resolution_summary TEXT,
    resolution_time_hours REAL CHECK (resolution_time_hours IS NULL OR resolution_time_hours >= 0),
    customer_sentiment TEXT,
    csat_score INTEGER CHECK (csat_score BETWEEN 0 AND 5),
    FOREIGN KEY (ticket_id) REFERENCES tickets(ticket_id)
);
''')
conn.commit()

In [25]:
clientes = df[['customer_id']].drop_duplicates()
conn.executemany('INSERT OR IGNORE INTO clientes (customer_id) VALUES (?)', clientes.values.tolist())
conn.commit()

In [26]:
cols_tickets = ['ticket_id','customer_id','created_at','customer_segment','sla_plan','region',
                'channel','product_area','issue_type','priority','status',
                'initial_message','agent_first_reply','platform','has_attachment','reopened']

tickets_data = df[cols_tickets].where(pd.notnull(df[cols_tickets]), None).values.tolist()

conn.executemany(f'''
    INSERT INTO tickets ({",".join(cols_tickets)})
    VALUES ({",".join(["?"]*len(cols_tickets))})
''', tickets_data)
conn.commit()

In [27]:
resueltos = df[df['resolution_time_hours'].notna()][
    ['ticket_id','resolution_summary','resolution_time_hours','customer_sentiment','csat_score']
]
conn.executemany('''
    INSERT INTO resoluciones (ticket_id, resolution_summary, resolution_time_hours, customer_sentiment, csat_score)
    VALUES (?,?,?,?,?)
''', resueltos.values.tolist())
conn.commit()

In [28]:
for tabla in ['clientes', 'tickets', 'resoluciones']:
    total = conn.execute(f'SELECT COUNT(*) FROM {tabla}').fetchone()[0]
    print(tabla, '->', total)

clientes -> 9999
tickets -> 100000
resoluciones -> 60113


In [29]:
try:
    conn.execute('''
        INSERT INTO tickets (ticket_id, customer_id, created_at, customer_segment, sla_plan,
                              channel, product_area, issue_type, priority, status)
        VALUES ('TCKT_TEST_001', 'CUST_NO_EXISTE', '2026-01-01', 'individual', 'standard',
                'email', 'billing', 'bug', 'low', 'open')
    ''')
    conn.commit()
    print("Se insertó (esto NO debería pasar)")
except sqlite3.IntegrityError as e:
    print("Rechazado correctamente:", e)

Rechazado correctamente: FOREIGN KEY constraint failed


In [30]:
try:
    conn.execute('''
        INSERT INTO tickets (ticket_id, customer_id, created_at, customer_segment, sla_plan,
                              channel, product_area, issue_type, priority, status)
        VALUES ('TCKT_TEST_002', 'CUST_00861', '2026-01-01', 'individual', 'standard',
                'email', 'billing', 'bug', 'urgentt', 'open')
    ''')
    conn.commit()
    print("Se insertó (esto NO debería pasar)")
except sqlite3.IntegrityError as e:
    print("Rechazado correctamente:", e)

Rechazado correctamente: CHECK constraint failed: priority IN ('low','medium','high','urgent')


In [31]:
try:
    conn.execute('''
        INSERT INTO tickets (ticket_id, customer_id, created_at, customer_segment, sla_plan,
                              channel, product_area, issue_type, priority, status)
        VALUES ('TCKT_TEST_003', 'CUST_00861', '2026-01-01', 'individual', 'standard',
                NULL, 'billing', 'bug', 'low', 'open')
    ''')
    conn.commit()
    print("Se insertó (esto NO debería pasar)")
except sqlite3.IntegrityError as e:
    print("Rechazado correctamente:", e)

Rechazado correctamente: NOT NULL constraint failed: tickets.channel


In [32]:
q1 = '''
SELECT t.priority,
       COUNT(*) AS total_resueltos,
       ROUND(AVG(r.resolution_time_hours), 2) AS horas_promedio
FROM tickets t
JOIN resoluciones r ON t.ticket_id = r.ticket_id
GROUP BY t.priority
ORDER BY horas_promedio;
'''
pd.read_sql_query(q1, conn)

,priority,total_resueltos,horas_promedio
0,urgent,3043,26.71
1,high,11882,31.56
2,medium,21358,43.28
3,low,23830,55.61


In [33]:
q2 = '''
SELECT product_area,
       status,
       COUNT(*) AS cantidad
FROM tickets
WHERE status IN ('open', 'in_progress', 'on_hold')
GROUP BY product_area, status
ORDER BY product_area, cantidad DESC;
'''
pd.read_sql_query(q2, conn)

,product_area,status,cantidad
0,analytics_dashboard,in_progress,2864
1,analytics_dashboard,on_hold,1440
2,analytics_dashboard,open,1418
3,api_integration,in_progress,2874
4,api_integration,open,1454
5,api_integration,on_hold,1430
6,billing,in_progress,2810
7,billing,on_hold,1504
8,billing,open,1463
9,data_export,in_progress,2801


In [34]:
q3 = '''
SELECT t.customer_id,
       COUNT(*) AS tickets_reabiertos
FROM tickets t
WHERE t.reopened = 1
GROUP BY t.customer_id
HAVING COUNT(*) > 1
ORDER BY tickets_reabiertos DESC
LIMIT 10;
'''
pd.read_sql_query(q3, conn)

,customer_id,tickets_reabiertos
0,CUST_04401,5
1,CUST_09587,4
2,CUST_09397,4
3,CUST_09244,4
4,CUST_08638,4
5,CUST_08151,4
6,CUST_07761,4
7,CUST_07355,4
8,CUST_05703,4
9,CUST_05661,4


In [35]:
plan_antes = pd.read_sql_query('''
    EXPLAIN QUERY PLAN
    SELECT customer_id, COUNT(*) FROM tickets
    WHERE reopened = 1
    GROUP BY customer_id
    HAVING COUNT(*) > 1;
''', conn)
print(plan_antes)

   id  parent  notused                        detail
0   6       0        0                  SCAN tickets
1  10       0        0  USE TEMP B-TREE FOR GROUP BY


In [36]:
conn.execute('CREATE INDEX idx_tickets_reopened ON tickets(reopened);')
conn.commit()

In [37]:
plan_despues = pd.read_sql_query('''
    EXPLAIN QUERY PLAN
    SELECT customer_id, COUNT(*) FROM tickets
    WHERE reopened = 1
    GROUP BY customer_id
    HAVING COUNT(*) > 1;
''', conn)
print(plan_despues)

   id  parent  notused                                             detail
0   7       0        0  SEARCH tickets USING INDEX idx_tickets_reopene...
1  12       0        0                       USE TEMP B-TREE FOR GROUP BY


## 10. Rendimiento

**Consulta evaluada:** tickets reabiertos por cliente (WHERE reopened=1 GROUP BY customer_id).

**Antes del índice:** `SCAN tickets` — recorrido completo de las 100,000 filas.

**Índice creado:** `CREATE INDEX idx_tickets_reopened ON tickets(reopened)`.

**Después del índice:** `SEARCH tickets USING INDEX idx_tickets_reopened` — acceso directo
a las ~5,045 filas con reopened=1, en vez de escanear toda la tabla.

**Nota:** el índice mejora el filtro, pero el GROUP BY sigue usando una estructura temporal
(B-TREE) porque no está indexado por customer_id. La mejora es real pero parcial.

In [38]:
try:
    with conn:  # 'with conn' maneja BEGIN/COMMIT/ROLLBACK automáticamente
        conn.execute('''
            UPDATE tickets SET status = 'resolved' WHERE ticket_id = 'TCKT_000010'
        ''')
        conn.execute('''
            INSERT INTO resoluciones (ticket_id, resolution_summary, resolution_time_hours,
                                       customer_sentiment, csat_score)
            VALUES ('TCKT_000010', 'Resuelto en prueba de transacción', 5.0, 'positive', 4)
        ''')
    print("Transacción exitosa: ambos cambios se aplicaron juntos")
except sqlite3.IntegrityError as e:
    print("Transacción revertida por completo:", e)

Transacción revertida por completo: UNIQUE constraint failed: resoluciones.ticket_id


In [39]:
try:
    with conn:
        conn.execute('''
            UPDATE tickets SET status = 'resolved' WHERE ticket_id = 'TCKT_000011'
        ''')
        conn.execute('''
            INSERT INTO resoluciones (ticket_id, resolution_summary, resolution_time_hours,
                                       customer_sentiment, csat_score)
            VALUES ('TCKT_000011', 'Prueba de fallo', 5.0, 'positive', 99)
        ''')  # csat_score=99 viola el CHECK (0 a 5) a propósito
except sqlite3.IntegrityError as e:
    print("Transacción revertida por completo:", e)

# Verificamos que el UPDATE también se revirtió (no quedó a medias):
verificacion = pd.read_sql_query(
    "SELECT status FROM tickets WHERE ticket_id = 'TCKT_000011'", conn)
print(verificacion)

Transacción revertida por completo: CHECK constraint failed: csat_score BETWEEN 0 AND 5
        status
0  in_progress


In [40]:
candidato = pd.read_sql_query('''
    SELECT t.ticket_id FROM tickets t
    LEFT JOIN resoluciones r ON t.ticket_id = r.ticket_id
    WHERE r.ticket_id IS NULL
    LIMIT 1
''', conn)
ticket_prueba = candidato['ticket_id'][0]
print("Vamos a usar:", ticket_prueba)

try:
    with conn:
        conn.execute("UPDATE tickets SET status = 'resolved' WHERE ticket_id = ?", (ticket_prueba,))
        conn.execute('''
            INSERT INTO resoluciones (ticket_id, resolution_summary, resolution_time_hours,
                                       customer_sentiment, csat_score)
            VALUES (?, 'Resuelto en prueba de transacción', 5.0, 'positive', 4)
        ''', (ticket_prueba,))
    print("Transacción exitosa: ambos cambios se aplicaron juntos")
except sqlite3.IntegrityError as e:
    print("Transacción revertida por completo:", e)

# Confirmamos que ambos cambios quedaron:
print(pd.read_sql_query("SELECT status FROM tickets WHERE ticket_id = ?", conn, params=(ticket_prueba,)))
print(pd.read_sql_query("SELECT * FROM resoluciones WHERE ticket_id = ?", conn, params=(ticket_prueba,)))

Vamos a usar: TCKT_000003
Transacción exitosa: ambos cambios se aplicaron juntos
     status
0  resolved
     ticket_id                 resolution_summary  resolution_time_hours  \
0  TCKT_000003  Resuelto en prueba de transacción                    5.0   

  customer_sentiment  csat_score  
0           positive           4  


## 12. Cierre

A partir del perfilado de synthetic_it_support_tickets.csv descubrimos que atributos que parecían
del cliente (segmento, plan SLA, región) en realidad varían por ticket, lo que definió
un modelo de 3 tablas (clientes, tickets, resoluciones) en vez de una tabla plana.
Implementamos integridad con PRIMARY KEY, FOREIGN KEY, NOT NULL y CHECK, y demostramos
con casos reales que la base de datos rechaza datos inválidos. La futura aplicación
deberá seguir estas mismas reglas al recibir tickets nuevos: cliente existente,
categorías válidas, y consistencia entre status y presencia de datos de resolución.